In [1]:
import numpy as np # linear algebra
import pandas as pd
import sqlite3


df_olist_customers = pd.read_csv('olist_customers_dataset.csv')
df_olist_sellers = pd.read_csv('olist_sellers_dataset.csv')
df_olist_order_reviews= pd.read_csv('olist_order_reviews_dataset.csv')
df_olist_order_items= pd.read_csv('olist_order_items_dataset.csv')
df_olist_products= pd.read_csv('olist_products_dataset.csv')
df_olist_geolocation= pd.read_csv('olist_geolocation_dataset.csv')
df_product_category_name_translation= pd.read_csv('product_category_name_translation.csv')
df_olist_orders = pd.read_csv('olist_orders_dataset.csv')
df_olist_order_payments= pd.read_csv('olist_order_payments_dataset.csv')

df_olist_customers.head()

from sqlalchemy import create_engine
engine = create_engine('sqlite://', echo=False)

# export the dataframe as a table 'playstore' to the sqlite engine
df_olist_customers.to_sql("olist_customers", con =engine)
df_olist_sellers.to_sql("olist_sellers", con =engine)
df_olist_order_reviews.to_sql("olist_order_reviews", con =engine)
df_olist_order_items.to_sql("olist_order_items", con =engine)
df_olist_products.to_sql("olist_products_dataset", con =engine)
df_olist_geolocation.to_sql("olist_geolocation", con =engine)
df_product_category_name_translation.to_sql("product_category_name_translation", con =engine)
df_olist_orders.to_sql("olist_orders", con =engine)
df_olist_order_payments.to_sql("olist_order_payments", con =engine)
df_olist_order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [2]:
sql='''

Select * from olist_customers
limit 5


''';


df_sql = pd.read_sql_query(sql,con=engine)
df_sql.head()

,index,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


**Query 1:** Count and Percentage of Orders Purchased in Jan 2018 with 5 Review Score

In [3]:
# Write and execute a SQL query to count the number of orders purchased in January 2018 that
# have a review score of 5 and calculate the percentage of such orders.
sql='''

SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
        COUNT(*) AS five_stars_count,
        COUNT(*) * 100.0 / (
            SELECT COUNT(*)
            FROM olist_orders o
            WHERE strftime('%Y-%m', o.order_purchase_timestamp) = '2018-01'
        ) AS five_stars_percentage
FROM olist_orders o
    JOIN olist_order_reviews orr ON o.order_id = orr.order_id
WHERE strftime('%Y-%m', o.order_purchase_timestamp) = '2018-01'
    AND orr.review_score = 5
GROUP BY month


''';


df_sql = pd.read_sql_query(sql,con=engine)
df_sql


,month,five_stars_count,five_stars_percentage
0,2018-01,4097,56.362636


**Query 2:** Customer Purchase Trend Year-on-Year

In [4]:
# Write and execute a SQL query to analyze the customer purchase trend year-on-year.
sql = '''

SELECT strftime('%Y', order_purchase_timestamp) AS year,
        COUNT(*) AS total_orders
FROM olist_orders
GROUP BY year
ORDER BY year DESC

'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,year,total_orders
0,2018,54011
1,2017,45101
2,2016,329


**Query 3:** Average Order Values of Customers

In [5]:
# Write and execute a SQL query to calculate the average order values of customers.
sql = '''

SELECT
    customer_id,
    AVG(order_total) AS avg_order_value
FROM (
    SELECT o.customer_id, oi.order_id, SUM(oi.price) AS order_total
    FROM olist_order_items oi
    JOIN olist_orders o ON oi.order_id = o.order_id
    GROUP BY o.customer_id, oi.order_id
    ) AS order_totals
GROUP BY customer_id
'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql


,customer_id,avg_order_value
0,00012a2ce6f8dcda20d059ce98491703,89.80
1,000161a058600d5901f007fab4c27140,54.90
2,0001fd6190edaaf884bcaf3d49edf079,179.99
3,0002414f95344307404f0ace7a26f1d5,149.90
4,000379cdec625522490c315e70c7a9fb,93.00
...,...,...
98661,fffcb937e9dd47a13f05ecb8290f4d3e,78.00
98662,fffecc9f79fd8c764f843e9951b11341,54.90
98663,fffeda5b6d849fbd39689bb92087f431,47.90
98664,ffff42319e9b2d713724ae527742af25,199.90


**Query 4:** Top 5 Cities with Highest Revenue from 2016 to 2018

In [6]:
# Write and execute a SQL query to find the top 5 cities with the highest revenue from 2016 to 2018.
sql = '''

SELECT s.seller_city,
        SUM(oi.price) AS total_revenue
FROM olist_order_items oi
JOIN olist_sellers s ON oi.seller_id = s.seller_id
JOIN olist_orders o ON oi.order_id = o.order_id
WHERE strftime('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY s.seller_city
ORDER BY total_revenue DESC
LIMIT 5

'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql


,seller_city,total_revenue
0,sao paulo,2702878.14
1,ibitinga,624592.94
2,curitiba,470759.82
3,rio de janeiro,358413.59
4,guarulhos,329494.38


**Query 5:** State Wise Revenue Table Between 2016 to 2018

In [7]:
# Write and execute a SQL query to create a state-wise revenue table between 2016 to 2018.
sql = '''

SELECT s.seller_state,
        SUM(oi.price) AS total_revenue
FROM olist_order_items oi
JOIN olist_sellers s ON oi.seller_id = s.seller_id
JOIN olist_orders o ON oi.order_id = o.order_id
WHERE strftime('%Y', o.order_purchase_timestamp) BETWEEN '2016' AND '2018'
GROUP BY s.seller_state
ORDER BY total_revenue DESC


'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,seller_state,total_revenue
0,SP,8.753396e+06
1,PR,1.261887e+06
2,MG,1.011565e+06
3,RJ,8.439842e+05
4,SC,6.324261e+05
5,RS,3.785595e+05
6,BA,2.855616e+05
7,DF,9.774948e+04
8,PE,9.149385e+04
9,GO,6.639921e+04


**Query 6:** Top Successful Sellers in Terms of Goods Sold, Revenue, and Customer Count

In [8]:
# Write and execute a SQL query to identify the top successful sellers in terms of the number
# of goods sold, total revenue, customer count, and sellers with the highest 5-star ratings.

sql = '''

SELECT COUNT(DISTINCT oi.order_id) AS goods_sold,
        SUM(oi.price) AS total_revenue,
        COUNT(DISTINCT o.customer_id) AS customer_count,
        COUNT(orr.review_id) AS five_star_reviews,
        s.seller_id
FROM olist_sellers s
    JOIN olist_order_items oi ON s.seller_id = oi.seller_id
    JOIN olist_orders o ON oi.order_id = o.order_id
    JOIN olist_order_reviews orr ON o.order_id = orr.order_id
WHERE orr.review_score = 5
GROUP BY s.seller_id
ORDER BY goods_sold DESC, total_revenue DESC, customer_count DESC, five_star_reviews DESC
LIMIT 5
'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql


,goods_sold,total_revenue,customer_count,five_star_reviews,seller_id
0,996,60943.55,996,1053,cc419e0650a3c5ba77189a1882b7556a
1,945,61309.91,945,1024,6560211a19b47992c3666cc44a7e94c0
2,865,94355.36,865,947,4a3ca9315b744ce9f8e9374361493884
3,841,60623.99,841,1096,1f50f920176fa81dab994f9023523100
4,771,90095.93,771,893,da8622b14eb17ae2831f4ac5b9dab84a


**Query 7:** Delivery Success Rate Across States

In [9]:
# Write and execute a SQL query to calculate the delivery success rate across different states.

sql = '''

SELECT s.seller_state,
        COUNT(*) * 1.0 / (
                      SELECT COUNT(*)
                      FROM olist_orders o
                      ) AS success_rate
FROM olist_orders o
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_sellers s ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
GROUP BY s.seller_state
ORDER BY success_rate DESC
'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql


,seller_state,success_rate
0,SP,0.790459
1,MG,0.086514
2,PR,0.085347
3,RJ,0.047113
4,SC,0.040225
5,RS,0.021812
6,DF,0.008880
7,BA,0.006275
8,GO,0.005109
9,PE,0.004475


**Query 8:** Preferred Form of Payment for Different Categories

In [10]:
# Write and execute a SQL query to find the preferred form of payment for different product categories.

sql = '''

SELECT
  p.product_category_name,
  op.payment_type,
  COUNT(*) AS payment_count
FROM olist_order_payments op
JOIN olist_orders o ON op.order_id = o.order_id
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_products_dataset p ON oi.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY p.product_category_name, op.payment_type
ORDER BY p.product_category_name, payment_count DESC;

'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql

,product_category_name,payment_type,payment_count
0,agro_industria_e_comercio,credit_card,145
1,agro_industria_e_comercio,boleto,60
2,agro_industria_e_comercio,voucher,42
3,agro_industria_e_comercio,debit_card,5
4,alimentos,credit_card,381
...,...,...,...
265,telefonia_fixa,debit_card,3
266,utilidades_domesticas,credit_card,5411
267,utilidades_domesticas,boleto,1326
268,utilidades_domesticas,voucher,505


**Query 9:** Distance Between Cities

In [11]:
# Write and execute a SQL query to calculate the distance between cities.

sql = '''

WITH city_coords AS (
    SELECT
        LOWER(TRIM(geolocation_city)) AS city,
        ROUND(AVG(geolocation_lat), 5) AS lat,
        ROUND(AVG(geolocation_lng), 5) AS lng
    FROM olist_geolocation
    WHERE geolocation_city IS NOT NULL
    GROUP BY geolocation_city
)

SELECT
    a.city AS city1,
    b.city AS city2,
    ROUND(
        6371 * ACOS(                                    -- Haversine formula for spherical distance
            COS(RADIANS(a.lat)) * COS(RADIANS(b.lat)) *
            COS(RADIANS(b.lng) - RADIANS(a.lng)) +
            SIN(RADIANS(a.lat)) * SIN(RADIANS(b.lat))
        )
    , 2) AS distance_km
FROM city_coords a
JOIN city_coords b ON a.city < b.city
ORDER BY distance_km DESC
LIMIT 100;


'''

df_sql = pd.read_sql_query(sql,con=engine)
df_sql


,city1,city2,distance_km
0,conquista d'oeste,santa lucia do piai,19946.34
1,nova lacerda,santa lucia do piai,19939.57
2,santa lucia do piai,vale de são domingos,19934.50
3,reserva do cabacal,santa lucia do piai,19934.20
4,santa lucia do piai,vale de sao domingos,19934.16
...,...,...,...
95,itanhanga,santa lucia do piai,19657.70
96,lucas do rio verde,santa lucia do piai,19653.46
97,barao de melgaco,santa lucia do piai,19652.28
98,porto dos gaúchos,santa lucia do piai,19639.63
